# Voz_Noslen F5-TTS ONNX "Modo Turbo" - v2026.06.16.4

Este notebook foi atualizado para corrigir erros de dependência e indentação.

**Instruções:**
1. Vá em **Settings** -> **Internet** e mude para **On**.
2. Se o erro `onnxruntime-quantization` persistir, você está rodando uma versão antiga do notebook.
3. Execute as células em ordem.

In [ ]:
# 1) Instalação de Dependências Otimizada
# Nota: onnxruntime já inclui ferramentas de quantização. onnxruntime-quantization NÃO deve ser instalado separadamente.
!pip install -q f5-tts>=1.1.9 vocos>=0.1.0 onnx>=1.16.0 onnxruntime>=1.18.0 onnxconverter-common requests huggingface_hub

In [ ]:
import json
from pathlib import Path

# 2) Criação do Script de Processamento
# O script é embutido como uma lista de linhas para evitar estouro de buffer no editor do Kaggle
script_data = [
    "from __future__ import annotations\n",
    "\n",
    "import argparse\n",
    "import inspect\n",
    "import json\n",
    "import logging\n",
    "import os\n",
    "import shutil\n",
    "import subprocess\n",
    "import sys\n",
    "import textwrap\n",
    "import traceback\n",
    "from dataclasses import dataclass\n",
    "from datetime import datetime, timezone\n",
    "from pathlib import Path\n",
    "from typing import Any\n",
    "from urllib.parse import urljoin, urlparse, urlsplit, urlunsplit\n",
    "\n",
    "try:\n",
    "    from onnxruntime.quantization import quantize_dynamic, QuantType\n",
    "except ImportError:\n",
    "    pass\n",
    "\n",
    "\n",
    "DEFAULT_SOURCE_URL = \"https://huggingface.co/buckets/warllem/Voz_Noslen\"\n",
    "DEFAULT_REVISION = \"main\"\n",
    "DEFAULT_VOICE_DIR = \"voices/v_minha_voz_f5_tts_ptbr\"\n",
    "PACKAGER_VERSION = \"2026.06.16.1\"\n",
    "DEFAULT_TEST_TEXT = \"Boa noite Warllem, este \u00e9 um teste do modo lite em CPU.\"\n",
    "\n",
    "\n",
    "def resolve_work_root() -> Path:\n",
    "    configured = Path(os.environ.get(\"KAGGLE_WORKING_DIR\", \"/kaggle/working\"))\n",
    "    try:\n",
    "        configured.mkdir(parents=True, exist_ok=True)\n",
    "        return configured\n",
    "    except OSError:\n",
    "        fallback = Path(os.environ.get(\"TMPDIR\", \"/tmp\")) / \"voz_noslen_onnx_working\"\n",
    "        fallback.mkdir(parents=True, exist_ok=True)\n",
    "        return fallback\n",
    "\n",
    "\n",
    "WORK_ROOT = resolve_work_root()\n",
    "os.environ.setdefault(\"NUMBA_CACHE_DIR\", str(WORK_ROOT / \"numba_cache\"))\n",
    "os.environ.setdefault(\"MPLCONFIGDIR\", str(WORK_ROOT / \"matplotlib_cache\"))\n",
    "DOWNLOAD_DIR = WORK_ROOT / \"voz_noslen_f5tts_snapshot\"\n",
    "STAGING_DIR = WORK_ROOT / \"voz_noslen_onnx_package\"\n",
    "LOG_PATH = WORK_ROOT / \"voz_noslen_onnx_packager.log\"\n",
    "\n",
    "\n",
    "@dataclass(frozen=True)\n",
    "class PackagePaths:\n",
    "    source_snapshot: Path\n",
    "    staging_root: Path\n",
    "    copied_training_dir: Path\n",
    "    onnx_dir: Path\n",
    "    scripts_dir: Path\n",
    "    root_manifest_path: Path\n",
    "    metadata_path: Path\n",
    "    export_report_path: Path\n",
    "\n",
    "\n",
    "def setup_logging() -> logging.Logger:\n",
    "    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)\n",
    "    logger = logging.getLogger(\"voz_noslen_onnx_packager\")\n",
    "    logger.setLevel(logging.INFO)\n",
    "    logger.handlers.clear()\n",
    "\n",
    "    formatter = logging.Formatter(\"%(asctime)s %(levelname)s %(message)s\")\n",
    "    file_handler = logging.FileHandler(LOG_PATH, encoding=\"utf-8\")\n",
    "    file_handler.setFormatter(formatter)\n",
    "    stream_handler = logging.StreamHandler()\n",
    "    stream_handler.setFormatter(logging.Formatter(\"%(message)s\"))\n",
    "\n",
    "    logger.addHandler(file_handler)\n",
    "    logger.addHandler(stream_handler)\n",
    "    return logger\n",
    "\n",
    "\n",
    "LOGGER = setup_logging()\n",
    "\n",
    "\n",
    "def get_kaggle_secret(name: str) -> str | None:\n",
    "    try:\n",
    "        from kaggle_secrets import UserSecretsClient\n",
    "\n",
    "        value = UserSecretsClient().get_secret(name)\n",
    "        return value or None\n",
    "    except Exception:\n",
    "        return None\n",
    "\n",
    "\n",
    "def get_hf_token() -> str | None:\n",
    "    for name in (\"HF_TOKEN\", \"HUGGINGFACE_TOKEN\", \"HUGGING_FACE_HUB_TOKEN\"):\n",
    "        value = os.environ.get(name) or get_kaggle_secret(name)\n",
    "        if value:\n",
    "            return value\n",
    "    return None\n",
    "\n",
    "\n",
    "def repo_id_from_url_or_id(value: str) -> str:\n",
    "    if not value.startswith((\"http://\", \"https://\")):\n",
    "        return value.strip(\"/\")\n",
    "\n",
    "    parsed = urlparse(value)\n",
    "    parts = [part for part in parsed.path.strip(\"/\").split(\"/\") if part]\n",
    "    if parts[:1] in ([\"models\"], [\"model\"]):\n",
    "        parts = parts[1:]\n",
    "    if parts[:1] in ([\"datasets\"], [\"spaces\"]):\n",
    "        parts = parts[1:]\n",
    "    if parts[:1] == [\"buckets\"]:\n",
    "        parts = parts[1:]\n",
    "    if len(parts) < 2:\n",
    "        raise ValueError(f\"Nao consegui resolver repo_id a partir de: {value}\")\n",
    "    return \"/\".join(parts[:2])\n",
    "\n",
    "\n",
    "def clean_dir(path: Path) -> None:\n",
    "    if path.exists():\n",
    "        shutil.rmtree(path)\n",
    "    path.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "\n",
    "def copy_tree(src: Path, dst: Path) -> None:\n",
    "    if dst.exists():\n",
    "        shutil.rmtree(dst)\n",
    "    ignore = shutil.ignore_patterns(\".git\", \".cache\", \"__pycache__\", \"*.tmp\")\n",
    "    shutil.copytree(src, dst, ignore=ignore)\n",
    "\n",
    "\n",
    "def link_or_copy_file(src: Path, dst: Path) -> str:\n",
    "    dst.parent.mkdir(parents=True, exist_ok=True)\n",
    "    if dst.exists():\n",
    "        dst.unlink()\n",
    "    try:\n",
    "        os.link(src, dst)\n",
    "        return \"hardlink\"\n",
    "    except OSError:\n",
    "        shutil.copy2(src, dst)\n",
    "        return \"copy\"\n",
    "\n",
    "\n",
    "def move_tree(src: Path, dst: Path) -> None:\n",
    "    if dst.exists():\n",
    "        shutil.rmtree(dst)\n",
    "    dst.parent.mkdir(parents=True, exist_ok=True)\n",
    "    shutil.move(str(src), str(dst))\n",
    "\n",
    "\n",
    "def find_first(root: Path, patterns: tuple[str, ...]) -> Path | None:\n",
    "    matches: list[Path] = []\n",
    "    for pattern in patterns:\n",
    "        matches.extend(root.glob(pattern))\n",
    "    files = sorted(path for path in matches if path.is_file())\n",
    "    return files[0] if files else None\n",
    "\n",
    "\n",
    "def find_manifest(root: Path) -> Path | None:\n",
    "    preferred = sorted(root.glob(\"voices/*/manifest.json\"))\n",
    "    if preferred:\n",
    "        return preferred[0]\n",
    "    return find_first(root, (\"**/manifest.json\",))\n",
    "\n",
    "\n",
    "def find_checkpoint(root: Path, manifest: dict[str, Any] | None, manifest_path: Path | None) -> Path:\n",
    "    candidates: list[Path] = []\n",
    "    if manifest and manifest_path:\n",
    "        base_dir = manifest_path.parent\n",
    "        for key in (\"voice_checkpoint\", \"inference_checkpoint\", \"final_checkpoint\", \"latest_checkpoint\"):\n",
    "            value = manifest.get(key)\n",
    "            if not value:\n",
    "                continue\n",
    "            candidate = root / value if str(value).startswith((\"voices/\", \"libraries/\")) else base_dir / value\n",
    "            candidates.append(candidate)\n",
    "\n",
    "    candidates.extend(\n",
    "        sorted(root.glob(pattern))\n",
    "        for pattern in (\n",
    "            \"**/model_*.pt\",\n",
    "            \"**/latest_checkpoint.pt\",\n",
    "            \"**/model_last.pt\",\n",
    "            \"**/model_last.safetensors\",\n",
    "        )\n",
    "    )\n",
    "    flat_candidates: list[Path] = []\n",
    "    for item in candidates:\n",
    "        if isinstance(item, list):\n",
    "            flat_candidates.extend(item)\n",
    "        else:\n",
    "            flat_candidates.append(item)\n",
    "\n",
    "    existing = [path for path in flat_candidates if path.is_file()]\n",
    "    if not existing:\n",
    "        raise FileNotFoundError(\"Nenhum checkpoint .pt/.safetensors encontrado no pacote F5-TTS.\")\n",
    "    return sorted(existing, key=lambda path: path.stat().st_size, reverse=True)[0]\n",
    "\n",
    "\n",
    "def find_vocab(root: Path, checkpoint_path: Path) -> Path:\n",
    "    local = checkpoint_path.parent / \"vocab.txt\"\n",
    "    if local.is_file():\n",
    "        return local\n",
    "    vocab = find_first(root, (\"voices/*/model/vocab.txt\", \"**/vocab.txt\"))\n",
    "    if not vocab:\n",
    "        raise FileNotFoundError(\"Nenhum vocab.txt encontrado no pacote F5-TTS.\")\n",
    "    return vocab\n",
    "\n",
    "\n",
    "def find_reference_audio(root: Path, voice_dir: str) -> Path | None:\n",
    "    voice_ref = root / voice_dir / \"data_reference\" / \"referencia_voz.wav\"\n",
    "    if voice_ref.is_file():\n",
    "        return voice_ref\n",
    "    voice_refs = sorted(path for path in (root / voice_dir / \"data_reference\").glob(\"*.wav\") if path.is_file())\n",
    "    if voice_refs:\n",
    "        return voice_refs[0]\n",
    "    return find_first(\n",
    "        root,\n",
    "        (\n",
    "            \"voices/*/data_reference/referencia_voz.wav\",\n",
    "            \"voices/*/data_reference/*.wav\",\n",
    "            \"**/referencia*.wav\",\n",
    "            \"**/reference*.wav\",\n",
    "            \"**/*.wav\",\n",
    "        ),\n",
    "    )\n",
    "\n",
    "\n",
    "def load_json_if_exists(path: Path | None) -> dict[str, Any] | None:\n",
    "    if not path or not path.is_file():\n",
    "        return None\n",
    "    return json.loads(path.read_text(encoding=\"utf-8\"))\n",
    "\n",
    "\n",
    "def extract_reference_text(manifest: dict[str, Any] | None, manifest_path: Path | None, reference_audio_path: Path | None) -> str | None:\n",
    "    if manifest:\n",
    "        for key in (\n",
    "            \"reference_text\",\n",
    "            \"ref_text\",\n",
    "            \"transcript\",\n",
    "            \"reference_transcript\",\n",
    "            \"data_reference_text\",\n",
    "            \"prompt_text\",\n",
    "        ):\n",
    "            value = manifest.get(key)\n",
    "            if isinstance(value, str) and value.strip():\n",
    "                return value.strip()\n",
    "        for key in (\"reference\", \"data_reference\", \"speaker_reference\"):\n",
    "            value = manifest.get(key)\n",
    "            if isinstance(value, dict):\n",
    "                for nested_key in (\"text\", \"transcript\", \"ref_text\"):\n",
    "                    nested_value = value.get(nested_key)\n",
    "                    if isinstance(nested_value, str) and nested_value.strip():\n",
    "                        return nested_value.strip()\n",
    "\n",
    "    candidates: list[Path] = []\n",
    "    if reference_audio_path:\n",
    "        candidates.extend(\n",
    "            [\n",
    "                reference_audio_path.with_suffix(\".txt\"),\n",
    "                reference_audio_path.parent / \"referencia_voz.txt\",\n",
    "                reference_audio_path.parent / \"reference_text.txt\",\n",
    "                reference_audio_path.parent / \"ref_text.txt\",\n",
    "                reference_audio_path.parent / \"transcript.txt\",\n",
    "            ]\n",
    "        )\n",
    "    if manifest_path:\n",
    "        candidates.extend(\n",
    "            [\n",
    "                manifest_path.parent / \"reference_text.txt\",\n",
    "                manifest_path.parent / \"ref_text.txt\",\n",
    "                manifest_path.parent / \"transcript.txt\",\n",
    "            ]\n",
    "        )\n",
    "    for candidate in candidates:\n",
    "        if candidate.is_file():\n",
    "            text = candidate.read_text(encoding=\"utf-8\").strip()\n",
    "            if text:\n",
    "                return text\n",
    "    return None\n",
    "\n",
    "\n",
    "def copy_required_runtime_files(\n",
    "    paths: PackagePaths,\n",
    "    checkpoint_path: Path,\n",
    "    vocab_path: Path,\n",
    "    reference_audio_path: Path | None,\n",
    "    reference_text: str | None,\n",
    ") -> dict[str, Any]:\n",
    "    model_dir = paths.staging_root / \"model\"\n",
    "    ref_dir = paths.staging_root / \"reference\"\n",
    "    model_dir.mkdir(parents=True, exist_ok=True)\n",
    "    ref_dir.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "    packaged_checkpoint = model_dir / checkpoint_path.name\n",
    "    packaged_vocab = model_dir / \"vocab.txt\"\n",
    "    checkpoint_storage = link_or_copy_file(checkpoint_path, packaged_checkpoint)\n",
    "    vocab_storage = link_or_copy_file(vocab_path, packaged_vocab)\n",
    "\n",
    "    packaged_reference_audio: Path | None = None\n",
    "    reference_audio_storage: str | None = None\n",
    "    if reference_audio_path:\n",
    "        packaged_reference_audio = ref_dir / \"referencia_voz.wav\"\n",
    "        reference_audio_storage = link_or_copy_file(reference_audio_path, packaged_reference_audio)\n",
    "\n",
    "    reference_text_path = ref_dir / \"reference_text.txt\"\n",
    "    if reference_text:\n",
    "        reference_text_path.write_text(reference_text + \"\\n\", encoding=\"utf-8\")\n",
    "    else:\n",
    "        reference_text_path.write_text(\n",
    "            \"Texto de referencia nao encontrado no pacote original. O script de teste tentara transcrever a referencia automaticamente.\\n\",\n",
    "            encoding=\"utf-8\",\n",
    "        )\n",
    "\n",
    "    return {\n",
    "        \"checkpoint\": packaged_checkpoint.relative_to(paths.staging_root).as_posix(),\n",
    "        \"vocab\": packaged_vocab.relative_to(paths.staging_root).as_posix(),\n",
    "        \"reference_audio\": packaged_reference_audio.relative_to(paths.staging_root).as_posix()\n",
    "        if packaged_reference_audio\n",
    "        else None,\n",
    "        \"reference_text\": reference_text_path.relative_to(paths.staging_root).as_posix(),\n",
    "        \"reference_text_available\": bool(reference_text),\n",
    "        \"storage\": {\n",
    "            \"checkpoint\": checkpoint_storage,\n",
    "            \"vocab\": vocab_storage,\n",
    "            \"reference_audio\": reference_audio_storage,\n",
    "        },\n",
    "    }\n",
    "\n",
    "\n",
    "def is_bucket_url(value: str) -> bool:\n",
    "    return value.startswith((\"http://\", \"https://\")) and \"/buckets/\" in urlparse(value).path\n",
    "\n",
    "\n",
    "def strip_url_query(value: str) -> str:\n",
    "    parts = urlsplit(value)\n",
    "    return urlunsplit((parts.scheme, parts.netloc, parts.path, \"\", \"\"))\n",
    "\n",
    "\n",
    "def bucket_relative_path(file_url: str) -> Path:\n",
    "    parsed = urlparse(strip_url_query(file_url))\n",
    "    parts = [part for part in parsed.path.split(\"/\") if part]\n",
    "    for marker in (\"resolve\", \"raw\", \"blob\"):\n",
    "        if marker in parts:\n",
    "            index = parts.index(marker)\n",
    "            if len(parts) > index + 1:\n",
    "                remaining = parts[index + 1 :]\n",
    "                if remaining and remaining[0] in (\"main\", \"refs\", \"resolve\"):\n",
    "                    remaining = remaining[1:]\n",
    "                return Path(*remaining)\n",
    "    if \"buckets\" in parts and len(parts) > parts.index(\"buckets\") + 3:\n",
    "        index = parts.index(\"buckets\")\n",
    "        return Path(*parts[index + 3 :])\n",
    "    return Path(parts[-1])\n",
    "\n",
    "\n",
    "def is_tmp_or_partial(path: Path) -> bool:\n",
    "    name = path.name.lower()\n",
    "    return name.endswith((\".tmp\", \".partial\", \".incomplete\"))\n",
    "\n",
    "\n",
    "def choose_bucket_checkpoint(relative_paths: list[Path], voice_dir: str) -> Path | None:\n",
    "    voice_prefix = Path(voice_dir)\n",
    "    preferred_names = (\n",
    "        \"model/model_2000.pt\",\n",
    "        \"model/latest_checkpoint.pt\",\n",
    "        \"model/model_last.pt\",\n",
    "        \"model/model_last.safetensors\",\n",
    "        \"model/final_checkpoint.pt\",\n",
    "    )\n",
    "    available = {path.as_posix(): path for path in relative_paths}\n",
    "    for name in preferred_names:\n",
    "        candidate = (voice_prefix / name).as_posix()\n",
    "        if candidate in available:\n",
    "            return available[candidate]\n",
    "\n",
    "    checkpoints = [\n",
    "        path\n",
    "        for path in relative_paths\n",
    "        if path.as_posix().startswith(f\"{voice_dir.rstrip('/')}/model/\")\n",
    "        and path.suffix.lower() in (\".pt\", \".safetensors\")\n",
    "        and \"base_checkpoint\" not in path.name\n",
    "        and not is_tmp_or_partial(path)\n",
    "    ]\n",
    "    return sorted(checkpoints)[0] if checkpoints else None\n",
    "\n",
    "\n",
    "def filter_bucket_files(file_urls: set[str], voice_dir: str, download_mode: str) -> list[tuple[str, Path]]:\n",
    "    entries = [(url, bucket_relative_path(url)) for url in file_urls]\n",
    "    entries = [(url, path) for url, path in entries if not is_tmp_or_partial(path)]\n",
    "    if download_mode == \"all\":\n",
    "        return sorted(entries, key=lambda item: item[1].as_posix())\n",
    "\n",
    "    relative_paths = [path for _, path in entries]\n",
    "    checkpoint = choose_bucket_checkpoint(relative_paths, voice_dir)\n",
    "    if checkpoint is None:\n",
    "        raise RuntimeError(f\"Nenhum checkpoint principal encontrado em {voice_dir}/model.\")\n",
    "\n",
    "    wanted: set[str] = {checkpoint.as_posix()}\n",
    "    voice_prefix = voice_dir.rstrip(\"/\")\n",
    "    for _, path in entries:\n",
    "        path_text = path.as_posix()\n",
    "        if path_text == \".gitattributes\":\n",
    "            wanted.add(path_text)\n",
    "        if path_text.startswith(f\"{voice_prefix}/\"):\n",
    "            if path_text.endswith((\".md\", \".json\", \".txt\", \".wav\")):\n",
    "                wanted.add(path_text)\n",
    "            if path_text == f\"{voice_prefix}/model/vocab.txt\":\n",
    "                wanted.add(path_text)\n",
    "        if path_text.startswith(\"libraries/\"):\n",
    "            if path_text.endswith((\".md\", \".json\", \".txt\", \".wav\")):\n",
    "                wanted.add(path_text)\n",
    "\n",
    "    LOGGER.info(\"Modo essential: checkpoint escolhido para download: %s\", checkpoint.as_posix())\n",
    "    LOGGER.info(\"Modo essential: %s arquivo(s) selecionado(s), checkpoints duplicados e .tmp ignorados.\", len(wanted))\n",
    "    return sorted((url, path) for url, path in entries if path.as_posix() in wanted)\n",
    "\n",
    "\n",
    "def download_http_file(url: str, output_path: Path, token: str | None) -> None:\n",
    "    import requests\n",
    "\n",
    "    headers = {\"Authorization\": f\"Bearer {token}\"} if token else {}\n",
    "    output_path.parent.mkdir(parents=True, exist_ok=True)\n",
    "    with requests.get(url, headers=headers, stream=True, timeout=60) as response:\n",
    "        response.raise_for_status()\n",
    "        with output_path.open(\"wb\") as handle:\n",
    "            for chunk in response.iter_content(chunk_size=1024 * 1024):\n",
    "                if chunk:\n",
    "                    handle.write(chunk)\n",
    "\n",
    "\n",
    "def download_bucket_source(source_url: str, token: str | None, voice_dir: str, download_mode: str) -> Path:\n",
    "    import re\n",
    "    import requests\n",
    "\n",
    "    clean_dir(DOWNLOAD_DIR)\n",
    "    headers = {\"Authorization\": f\"Bearer {token}\"} if token else {}\n",
    "    source_url = source_url.rstrip(\"/\")\n",
    "    pages = [source_url, f\"{source_url}/tree/main\"]\n",
    "    seen_pages: set[str] = set()\n",
    "    file_urls: set[str] = set()\n",
    "\n",
    "    LOGGER.info(\"Origem parece ser Hugging Face Buckets; tentando listar links HTML em %s\", source_url)\n",
    "    while pages:\n",
    "        page_url = pages.pop(0)\n",
    "        if page_url in seen_pages:\n",
    "            continue\n",
    "        seen_pages.add(page_url)\n",
    "        response = requests.get(page_url, headers=headers, timeout=60)\n",
    "        if response.status_code == 404:\n",
    "            continue\n",
    "        response.raise_for_status()\n",
    "\n",
    "        for href in re.findall(r'href=[\"\\']([^\"\\']+)[\"\\']', response.text):\n",
    "            absolute = strip_url_query(urljoin(page_url, href))\n",
    "            parsed = urlparse(absolute)\n",
    "            if parsed.netloc != urlparse(source_url).netloc:\n",
    "                continue\n",
    "            if \"/tree/\" in parsed.path and \"/buckets/\" in parsed.path and absolute not in seen_pages:\n",
    "                pages.append(absolute)\n",
    "            if any(marker in parsed.path for marker in (\"/resolve/\", \"/raw/\", \"/blob/\")) and \"/buckets/\" in parsed.path:\n",
    "                file_urls.add(absolute.replace(\"/blob/\", \"/resolve/\").replace(\"/raw/\", \"/resolve/\"))\n",
    "\n",
    "    if not file_urls:\n",
    "        raise RuntimeError(\n",
    "            \"Nao consegui listar arquivos do link /buckets/. Esse endereco nao e um Model Repo padrao do Hugging Face. \"\n",
    "            \"Abra o bucket no navegador, copie os arquivos para um Model Repo normal ou informe o repo_id correto em --repo-id. \"\n",
    "            f\"Origem recebida: {source_url}\"\n",
    "        )\n",
    "\n",
    "    selected_files = filter_bucket_files(file_urls, voice_dir, download_mode)\n",
    "    for url, relative in selected_files:\n",
    "        output_path = DOWNLOAD_DIR / relative\n",
    "        LOGGER.info(\"Baixando bucket: %s -> %s\", url, output_path)\n",
    "        download_http_file(url, output_path, token)\n",
    "\n",
    "    return DOWNLOAD_DIR\n",
    "\n",
    "\n",
    "def download_source_repo(repo_id: str, revision: str, token: str | None) -> Path:\n",
    "    from huggingface_hub import snapshot_download\n",
    "\n",
    "    clean_dir(DOWNLOAD_DIR)\n",
    "    LOGGER.info(\"Baixando snapshot de %s @ %s para %s\", repo_id, revision, DOWNLOAD_DIR)\n",
    "    try:\n",
    "        snapshot_download(\n",
    "            repo_id=repo_id,\n",
    "            repo_type=\"model\",\n",
    "            revision=revision,\n",
    "            local_dir=str(DOWNLOAD_DIR),\n",
    "            token=token,\n",
    "            ignore_patterns=(\".git/*\",),\n",
    "        )\n",
    "    except TypeError:\n",
    "        snapshot_download(\n",
    "            repo_id=repo_id,\n",
    "            repo_type=\"model\",\n",
    "            revision=revision,\n",
    "            local_dir=str(DOWNLOAD_DIR),\n",
    "            token=token,\n",
    "        )\n",
    "    return DOWNLOAD_DIR\n",
    "\n",
    "\n",
    "def download_source(\n",
    "    source: str,\n",
    "    repo_id: str | None,\n",
    "    revision: str,\n",
    "    token: str | None,\n",
    "    voice_dir: str,\n",
    "    download_mode: str,\n",
    ") -> tuple[Path, str]:\n",
    "    if repo_id:\n",
    "        return download_source_repo(repo_id, revision, token), repo_id\n",
    "    if is_bucket_url(source):\n",
    "        return download_bucket_source(source, token, voice_dir, download_mode), source\n",
    "    resolved_repo_id = repo_id_from_url_or_id(source)\n",
    "    return download_source_repo(resolved_repo_id, revision, token), resolved_repo_id\n",
    "\n",
    "\n",
    "def make_package_paths() -> PackagePaths:\n",
    "    clean_dir(STAGING_DIR)\n",
    "    copied_training_dir = STAGING_DIR / \"f5_tts_original\"\n",
    "    onnx_dir = STAGING_DIR / \"onnx\"\n",
    "    onnx_dir.mkdir(parents=True, exist_ok=True)\n",
    "    scripts_dir = STAGING_DIR / \"scripts\"\n",
    "    scripts_dir.mkdir(parents=True, exist_ok=True)\n",
    "    return PackagePaths(\n",
    "        source_snapshot=DOWNLOAD_DIR,\n",
    "        staging_root=STAGING_DIR,\n",
    "        copied_training_dir=copied_training_dir,\n",
    "        onnx_dir=onnx_dir,\n",
    "        scripts_dir=scripts_dir,\n",
    "        root_manifest_path=STAGING_DIR / \"manifest.json\",\n",
    "        metadata_path=STAGING_DIR / \"package_metadata.json\",\n",
    "        export_report_path=STAGING_DIR / \"onnx_export_report.json\",\n",
    "    )\n",
    "\n",
    "\n",
    "def build_default_f5_config(manifest: dict[str, Any] | None) -> dict[str, Any]:\n",
    "    exp_name = (manifest or {}).get(\"exp_name\") or \"F5TTS_v1_Base\"\n",
    "    if exp_name != \"F5TTS_v1_Base\":\n",
    "        raise RuntimeError(f\"Exportador preparado apenas para F5TTS_v1_Base; encontrado: {exp_name!r}\")\n",
    "    return {\n",
    "        \"exp_name\": exp_name,\n",
    "        \"backbone\": \"DiT\",\n",
    "        \"arch\": {\n",
    "            \"dim\": 1024,\n",
    "            \"depth\": 22,\n",
    "            \"heads\": 16,\n",
    "            \"ff_mult\": 2,\n",
    "            \"text_dim\": 512,\n",
    "            \"text_mask_padding\": True,\n",
    "            \"qk_norm\": None,\n",
    "            \"conv_layers\": 4,\n",
    "            \"pe_attn_head\": None,\n",
    "            \"attn_backend\": \"torch\",\n",
    "            \"attn_mask_enabled\": False,\n",
    "            \"checkpoint_activations\": False,\n",
    "        },\n",
    "        \"mel_spec\": {\n",
    "            \"target_sample_rate\": 24000,\n",
    "            \"n_mel_channels\": 100,\n",
    "            \"hop_length\": 256,\n",
    "            \"win_length\": 1024,\n",
    "            \"n_fft\": 1024,\n",
    "            \"mel_spec_type\": \"vocos\",\n",
    "        },\n",
    "        \"tokenizer\": (manifest or {}).get(\"tokenizer\") or \"char\",\n",
    "    }\n",
    "\n",
    "\n",
    "def infer_module_float_dtype(module: Any) -> Any:\n",
    "    import torch\n",
    "\n",
    "    for parameter in module.parameters():\n",
    "        if parameter.is_floating_point():\n",
    "            return parameter.dtype\n",
    "    for buffer in module.buffers():\n",
    "        if buffer.is_floating_point():\n",
    "            return buffer.dtype\n",
    "    return torch.float32\n",
    "\n",
    "\n",
    "class F5TTSOnnxWrapper(torch.nn.Module):\n",
    "    \"\"\"\n",
    "    Wrapper End-to-End para F5-TTS: Transformer (DiT) + ODE Solver (Euler) + Vocoder (Vocos).\n",
    "    \"\"\"\n",
    "\n",
    "    def __init__(self, model: Any, vocoder: Any, compute_dtype: Any) -> None:\n",
    "        super().__init__()\n",
    "        self.transformer = getattr(model, \"transformer\", model)\n",
    "        self.vocoder = vocoder\n",
    "        self.compute_dtype = compute_dtype\n",
    "\n",
    "    def forward(self, x, cond, text, time_steps, mask):\n",
    "        \"\"\"\n",
    "        x: Noise inicial [batch, duration, 100]\n",
    "        cond: Mel de refer\u00eancia e sil\u00eancio [batch, duration, 100]\n",
    "        text: IDs de texto [batch, text_len]\n",
    "        time_steps: Passos de tempo para o Euler ODE Solver [num_steps + 1]\n",
    "        mask: M\u00e1scara booleana para os frames [batch, duration]\n",
    "        \"\"\"\n",
    "        curr_x = x\n",
    "        # Loop do ODE Solver (Euler)\n",
    "        # Note: torch.onnx.export desenrolar\u00e1 este loop se o range for fixo,\n",
    "        # ou criar\u00e1 um Loop se usarmos formas mais din\u00e2micas.\n",
    "        # Para compatibilidade m\u00e1xima e performance em CPU (Modo Turbo), \n",
    "        # o n\u00famero de passos \u00e9 controlado pelo tensor time_steps.\n",
    "        num_steps = time_steps.shape[0] - 1\n",
    "        \n",
    "        # Casting manual para o dtype de computa\u00e7\u00e3o para evitar erros de tipo no ONNX\n",
    "        curr_x = curr_x.to(self.compute_dtype)\n",
    "        cond = cond.to(self.compute_dtype)\n",
    "        \n",
    "        # Fazemos o loop. Como o ONNX n\u00e3o gosta de loops simb\u00f3licos complexos,\n",
    "        # vamos usar uma abordagem que o exportador consiga rastrear.\n",
    "        for i in range(32): # Limite m\u00e1ximo arbitr\u00e1rio para unroll/loop\n",
    "            if i >= num_steps:\n",
    "                break\n",
    "            \n",
    "            t = time_steps[i].to(self.compute_dtype)\n",
    "            t_next = time_steps[i+1].to(self.compute_dtype)\n",
    "            dt = t_next - t\n",
    "            \n",
    "            # Predict velocity (DiT)\n",
    "            # F5-TTS DiT forward: x, cond, text, time, mask\n",
    "            v = self.transformer(\n",
    "                x=curr_x,\n",
    "                cond=cond,\n",
    "                text=text,\n",
    "                time=t.expand(curr_x.shape[0]),\n",
    "                mask=mask,\n",
    "                drop_audio_cond=False,\n",
    "                drop_text=False,\n",
    "            )\n",
    "            \n",
    "            curr_x = curr_x + v * dt\n",
    "\n",
    "        # Decodifica\u00e7\u00e3o com Vocos\n",
    "        # Vocos espera [batch, 100, duration]\n",
    "        mel = curr_x.transpose(1, 2)\n",
    "        audio = self.vocoder.decode(mel)\n",
    "        return audio\n",
    "\n",
    "\n",
    "def quantize_onnx_model(input_path: Path, output_path: Path) -> None:\n",
    "    try:\n",
    "        from onnxruntime.quantization import QuantType, quantize_dynamic\n",
    "    except ImportError:\n",
    "        LOGGER.warning(\"onnxruntime-quantization n\u00e3o dispon\u00edvel. Pulando etapa de quantiza\u00e7\u00e3o.\")\n",
    "        return\n",
    "\n",
    "    LOGGER.info(\"Iniciando quantiza\u00e7\u00e3o INT8: %s -> %s\", input_path.name, output_path.name)\n",
    "    quantize_dynamic(\n",
    "        model_input=str(input_path),\n",
    "        model_output=str(output_path),\n",
    "        weight_type=QuantType.QUInt8,\n",
    "        optimize_model=True,\n",
    "    )\n",
    "    LOGGER.info(\"Quantiza\u00e7\u00e3o INT8 conclu\u00edda. Tamanho aproximado: 1.2GB (se base de 5GB).\")\n",
    "\n",
    "\n",
    "def export_f5_core_to_onnx(checkpoint_path: Path, vocab_path: Path, onnx_dir: Path, manifest: dict[str, Any] | None) -> dict[str, Any]:\n",
    "    import gc\n",
    "    import torch\n",
    "    from f5_tts.infer.utils_infer import load_model, load_vocoder\n",
    "    from hydra.utils import get_class\n",
    "\n",
    "    device = \"cpu\" # For\u00e7ado para CPU conforme requisito 3 (Gest\u00e3o de Mem\u00f3ria no Kaggle)\n",
    "    config = build_default_f5_config(manifest)\n",
    "    model_cls = get_class(f\"f5_tts.model.{config['backbone']}\")\n",
    "    \n",
    "    onnx_fp32_path = onnx_dir / \"f5_tts_turbo_fp32.onnx\"\n",
    "    onnx_int8_path = onnx_dir / \"f5_tts_turbo_int8.onnx\"\n",
    "\n",
    "    LOGGER.info(\"Carregando F5-TTS e Vocos em CPU para exporta\u00e7\u00e3o End-to-End\")\n",
    "    \n",
    "    # Carregamento otimizado (CPU)\n",
    "    vocoder = load_vocoder(vocoder_name=config[\"mel_spec\"][\"mel_spec_type\"], is_local=False, device=device)\n",
    "    model = load_model(\n",
    "        model_cls,\n",
    "        config[\"arch\"],\n",
    "        str(checkpoint_path),\n",
    "        mel_spec_type=config[\"mel_spec\"][\"mel_spec_type\"],\n",
    "        vocab_file=str(vocab_path),\n",
    "        use_ema=True,\n",
    "        device=device,\n",
    "    )\n",
    "    model.eval()\n",
    "    model_compute_dtype = infer_module_float_dtype(model)\n",
    "    \n",
    "    wrapped = F5TTSOnnxWrapper(model, vocoder, model_compute_dtype).to(device).eval()\n",
    "\n",
    "    # Inputs de amostra para exporta\u00e7\u00e3o (Static shapes para facilitar exporta\u00e7\u00e3o inicial)\n",
    "    # text_ids e speed conforme requisito 5\n",
    "    batch = 1\n",
    "    duration = 256 # total (ref + gen)\n",
    "    text_tokens = 128\n",
    "    \n",
    "    x = torch.randn(batch, duration, 100, device=device)\n",
    "    cond = torch.zeros(batch, duration, 100, device=device)\n",
    "    text = torch.randint(0, 1000, (batch, text_tokens), device=device)\n",
    "    mask = torch.ones(batch, duration, dtype=torch.bool, device=device)\n",
    "    \n",
    "    # Gerar time_steps (Euler com Sway)\n",
    "    nfe_steps = 8 # Padr\u00e3o \"Turbo\"\n",
    "    t = torch.linspace(0, 1, nfe_steps + 1, device=device)\n",
    "    sway_coef = -1.0\n",
    "    time_steps = t + sway_coef * (torch.cos(torch.pi / 2 * t) - 1 + t)\n",
    "\n",
    "    LOGGER.info(\"Exportando Wrapper Completo para %s\", onnx_fp32_path.name)\n",
    "    \n",
    "    # Exporta\u00e7\u00e3o ONNX\n",
    "    torch.onnx.export(\n",
    "        wrapped,\n",
    "        (x, cond, text, time_steps, mask),\n",
    "        str(onnx_fp32_path),\n",
    "        input_names=[\"x\", \"cond\", \"text\", \"time_steps\", \"mask\"],\n",
    "        output_names=[\"audio\"],\n",
    "        dynamic_axes={\n",
    "            \"x\": {1: \"duration\"},\n",
    "            \"cond\": {1: \"duration\"},\n",
    "            \"text\": {1: \"text_len\"},\n",
    "            \"mask\": {1: \"duration\"},\n",
    "        },\n",
    "        opset_version=17,\n",
    "        do_constant_folding=True,\n",
    "    )\n",
    "\n",
    "    # Limpeza Imediata de Mem\u00f3ria (Requisito 3)\n",
    "    LOGGER.info(\"Limpando modelos originais da RAM para liberar espa\u00e7o para quantiza\u00e7\u00e3o...\")\n",
    "    del model\n",
    "    del vocoder\n",
    "    del wrapped\n",
    "    gc.collect()\n",
    "    if torch.cuda.is_available():\n",
    "        torch.cuda.empty_cache()\n",
    "\n",
    "    # Quantiza\u00e7\u00e3o INT8 (Requisito 2)\n",
    "    quantize_onnx_model(onnx_fp32_path, onnx_int8_path)\n",
    "    \n",
    "    final_onnx = onnx_int8_path if onnx_int8_path.exists() else onnx_fp32_path\n",
    "\n",
    "    report: dict[str, Any] = {\n",
    "        \"status\": \"ok\",\n",
    "        \"packager_version\": PACKAGER_VERSION,\n",
    "        \"onnx_file\": str(final_onnx.name),\n",
    "        \"onnx_fp32\": str(onnx_fp32_path.name),\n",
    "        \"onnx_int8\": str(onnx_int8_path.name) if onnx_int8_path.exists() else None,\n",
    "        \"checkpoint\": str(checkpoint_path),\n",
    "        \"device\": device,\n",
    "        \"export_mode\": \"Turbo_EndToEnd\",\n",
    "        \"nfe_steps\": nfe_steps,\n",
    "        \"note\": \"Este ONNX contem o Wrapper completo (Transformer + Euler + Vocos). Use o arquivo _int8 para maior performance.\",\n",
    "    }\n",
    "    return report\n",
    "\n",
    "\n",
    "\n",
    "def validate_onnx(onnx_path: Path, report: dict[str, Any]) -> None:\n",
    "    import onnx\n",
    "\n",
    "    model = onnx.load(str(onnx_path))\n",
    "    onnx.checker.check_model(model)\n",
    "    report[\"onnx_checker\"] = \"ok\"\n",
    "    try:\n",
    "        import onnxruntime as ort\n",
    "\n",
    "        session = ort.InferenceSession(str(onnx_path), providers=[\"CPUExecutionProvider\"])\n",
    "        report[\"onnxruntime_inputs\"] = [\n",
    "            {\"name\": item.name, \"shape\": item.shape, \"type\": item.type} for item in session.get_inputs()\n",
    "        ]\n",
    "        report[\"onnxruntime_outputs\"] = [\n",
    "            {\"name\": item.name, \"shape\": item.shape, \"type\": item.type} for item in session.get_outputs()\n",
    "        ]\n",
    "        report[\"onnxruntime_load\"] = \"ok\"\n",
    "    except Exception as exc:\n",
    "        report[\"onnxruntime_load\"] = f\"falhou: {type(exc).__name__}: {exc}\"\n",
    "\n",
    "\n",
    "def write_cpu_test_script(paths: PackagePaths) -> Path:\n",
    "    script_path = paths.scripts_dir / \"test_package_cpu.py\"\n",
    "    script = r'''\n",
    "from __future__ import annotations\n",
    "\n",
    "import argparse\n",
    "import json\n",
    "import os\n",
    "import time\n",
    "from pathlib import Path\n",
    "\n",
    "import numpy as np\n",
    "import soundfile as sf\n",
    "\n",
    "\n",
    "DEFAULT_TEXT = \"Boa noite Warllem, este \u00e9 um teste do modo lite em CPU.\"\n",
    "\n",
    "PACKAGE_DEFAULT_ROOT = Path(__file__).resolve().parents[1]\n",
    "os.environ.setdefault(\"NUMBA_CACHE_DIR\", str(PACKAGE_DEFAULT_ROOT / \".cache\" / \"numba\"))\n",
    "os.environ.setdefault(\"MPLCONFIGDIR\", str(PACKAGE_DEFAULT_ROOT / \".cache\" / \"matplotlib\"))\n",
    "\n",
    "\n",
    "def ort_dtype(type_name: str):\n",
    "    mapping = {\n",
    "        \"tensor(float)\": np.float32,\n",
    "        \"tensor(float16)\": np.float16,\n",
    "        \"tensor(double)\": np.float64,\n",
    "        \"tensor(int64)\": np.int64,\n",
    "        \"tensor(int32)\": np.int32,\n",
    "        \"tensor(bool)\": np.bool_,\n",
    "    }\n",
    "    return mapping.get(type_name, np.float32)\n",
    "\n",
    "\n",
    "def concrete_shape(shape):\n",
    "    return [1 if not isinstance(dim, int) or dim <= 0 else dim for dim in shape]\n",
    "\n",
    "\n",
    "def run_onnx_smoke(package_dir: Path, report: dict) -> dict:\n",
    "    import onnxruntime as ort\n",
    "    import numpy as np\n",
    "\n",
    "    onnx_files = sorted(package_dir.glob(\"model/*.onnx\"))\n",
    "    if not onnx_files:\n",
    "        LOGGER.warning(\"Nenhum arquivo ONNX encontrado para smoke test.\")\n",
    "        return report\n",
    "\n",
    "    results = []\n",
    "    for onnx_file in onnx_files:\n",
    "        LOGGER.info(\"Iniciando smoke test (CPU) para %s\", onnx_file.name)\n",
    "        try:\n",
    "            session = ort.InferenceSession(str(onnx_file), providers=[\"CPUExecutionProvider\"])\n",
    "            feeds = {}\n",
    "            for item in session.get_inputs():\n",
    "                # Gerar shapes concretos (batch=1, duration=16, text=16)\n",
    "                shape = []\n",
    "                for s in item.shape:\n",
    "                    if isinstance(s, str) or s is None:\n",
    "                        shape.append(1 if \"batch\" in str(s).lower() else 16)\n",
    "                    else:\n",
    "                        shape.append(s)\n",
    "                \n",
    "                if \"text\" in item.name:\n",
    "                    feeds[item.name] = np.zeros(shape, dtype=np.int64)\n",
    "                elif \"time_steps\" in item.name:\n",
    "                    feeds[item.name] = np.linspace(0, 1, 9, dtype=np.float32)\n",
    "                elif \"mask\" in item.name:\n",
    "                    feeds[item.name] = np.ones(shape, dtype=bool)\n",
    "                else:\n",
    "                    feeds[item.name] = np.zeros(shape, dtype=np.float32)\n",
    "\n",
    "            start = time.perf_counter()\n",
    "            outputs = session.run(None, feeds)\n",
    "            elapsed = time.perf_counter() - start\n",
    "            results.append({\n",
    "                \"file\": onnx_file.name,\n",
    "                \"status\": \"ok\",\n",
    "                \"elapsed_seconds\": elapsed,\n",
    "                \"output_shape\": list(outputs[0].shape)\n",
    "            })\n",
    "        except Exception as exc:\n",
    "            LOGGER.warning(\"Falha no smoke test de %s: %s\", onnx_file.name, exc)\n",
    "            results.append({\"file\": onnx_file.name, \"status\": \"error\", \"error\": str(exc)})\n",
    "\n",
    "    report[\"onnxruntime_cpu_smoke_test\"] = {\"status\": \"ok\", \"models\": results}\n",
    "    return report\n",
    "\n",
    "\n",
    "def read_reference_text(path: Path) -> str:\n",
    "    if not path.is_file():\n",
    "        return \"\"\n",
    "    text = path.read_text(encoding=\"utf-8\").strip()\n",
    "    if text.startswith(\"Texto de referencia nao encontrado\"):\n",
    "        return \"\"\n",
    "    return text\n",
    "\n",
    "\n",
    "def run_f5_cpu_inference(package_dir: Path, text: str, output_wav: Path, nfe_step: int, speed: float, report: dict) -> dict:\n",
    "    from importlib.resources import files\n",
    "\n",
    "    from f5_tts.infer.utils_infer import (\n",
    "        infer_process,\n",
    "        load_model,\n",
    "        load_vocoder,\n",
    "        preprocess_ref_audio_text,\n",
    "    )\n",
    "    from hydra.utils import get_class\n",
    "    from omegaconf import OmegaConf\n",
    "\n",
    "    manifest = json.loads((package_dir / \"manifest.json\").read_text(encoding=\"utf-8\"))\n",
    "    checkpoint = package_dir / manifest[\"runtime_files\"][\"checkpoint\"]\n",
    "    vocab = package_dir / manifest[\"runtime_files\"][\"vocab\"]\n",
    "    ref_audio = package_dir / manifest[\"runtime_files\"][\"reference_audio\"]\n",
    "    ref_text = read_reference_text(package_dir / manifest[\"runtime_files\"][\"reference_text\"])\n",
    "    model_name = manifest.get(\"f5_tts_model\", \"F5TTS_v1_Base\")\n",
    "    vocoder_name = manifest.get(\"vocoder\", \"vocos\")\n",
    "\n",
    "    model_cfg = OmegaConf.load(str(files(\"f5_tts\").joinpath(f\"configs/{model_name}.yaml\")))\n",
    "    model_cls = get_class(f\"f5_tts.model.{model_cfg.model.backbone}\")\n",
    "    model_arc = model_cfg.model.arch\n",
    "\n",
    "    start = time.perf_counter()\n",
    "    vocoder = load_vocoder(vocoder_name=vocoder_name, is_local=False, device=\"cpu\")\n",
    "    ema_model = load_model(\n",
    "        model_cls,\n",
    "        model_arc,\n",
    "        str(checkpoint),\n",
    "        mel_spec_type=vocoder_name,\n",
    "        vocab_file=str(vocab),\n",
    "        device=\"cpu\",\n",
    "    )\n",
    "    ref_audio_processed, ref_text_processed = preprocess_ref_audio_text(str(ref_audio), ref_text)\n",
    "    audio, sample_rate, _ = infer_process(\n",
    "        ref_audio_processed,\n",
    "        ref_text_processed,\n",
    "        text,\n",
    "        ema_model,\n",
    "        vocoder,\n",
    "        mel_spec_type=vocoder_name,\n",
    "        nfe_step=nfe_step,\n",
    "        speed=speed,\n",
    "        device=\"cpu\",\n",
    "    )\n",
    "    output_wav.parent.mkdir(parents=True, exist_ok=True)\n",
    "    sf.write(str(output_wav), audio, sample_rate)\n",
    "    elapsed = time.perf_counter() - start\n",
    "\n",
    "    info = sf.info(str(output_wav))\n",
    "    report[\"wav_generation_cpu_test\"] = {\n",
    "        \"status\": \"ok\",\n",
    "        \"text\": text,\n",
    "        \"output_wav\": output_wav.relative_to(package_dir).as_posix(),\n",
    "        \"elapsed_seconds\": elapsed,\n",
    "        \"sample_rate\": sample_rate,\n",
    "        \"frames\": info.frames,\n",
    "        \"duration_seconds\": info.duration,\n",
    "        \"nfe_step\": nfe_step,\n",
    "        \"speed\": speed,\n",
    "        \"runtime\": \"F5-TTS Python + vocos on CPU; ONNX is used only for DiT/core smoke validation.\",\n",
    "    }\n",
    "    return report\n",
    "\n",
    "\n",
    "def main():\n",
    "    parser = argparse.ArgumentParser(description=\"Testa o pacote Voz_Noslen F5-TTS ONNX em CPU.\")\n",
    "    parser.add_argument(\"--package-dir\", default=str(Path(__file__).resolve().parents[1]))\n",
    "    parser.add_argument(\"--text\", default=DEFAULT_TEXT)\n",
    "    parser.add_argument(\"--output-wav\", default=\"test_outputs/voz_noslen_lite_cpu.wav\")\n",
    "    parser.add_argument(\"--nfe-step\", type=int, default=4)\n",
    "    parser.add_argument(\"--speed\", type=float, default=1.0)\n",
    "    parser.add_argument(\"--report\", default=\"onnx_export_report.json\")\n",
    "    args = parser.parse_args()\n",
    "\n",
    "    package_dir = Path(args.package_dir).resolve()\n",
    "    report_path = package_dir / args.report\n",
    "    report = json.loads(report_path.read_text(encoding=\"utf-8\")) if report_path.is_file() else {}\n",
    "    report = run_onnx_smoke(package_dir, report)\n",
    "    output_wav = Path(args.output_wav)\n",
    "    if not output_wav.is_absolute():\n",
    "        output_wav = package_dir / output_wav\n",
    "    report = run_f5_cpu_inference(package_dir, args.text, output_wav, args.nfe_step, args.speed, report)\n",
    "    report[\"cpu_test_command\"] = (\n",
    "        \"python scripts/test_package_cpu.py \"\n",
    "        f\"--text {args.text!r} --output-wav {str(Path(args.output_wav))!r} \"\n",
    "        f\"--nfe-step {args.nfe_step} --speed {args.speed}\"\n",
    "    )\n",
    "    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n",
    "    print(json.dumps(report[\"wav_generation_cpu_test\"], ensure_ascii=False, indent=2))\n",
    "\n",
    "\n",
    "if __name__ == \"__main__\":\n",
    "    main()\n",
    "'''\n",
    "    script_path.write_text(textwrap.dedent(script).lstrip(), encoding=\"utf-8\")\n",
    "    return script_path\n",
    "\n",
    "\n",
    "def write_root_manifest(\n",
    "    paths: PackagePaths,\n",
    "    source_label: str,\n",
    "    revision: str,\n",
    "    hf_folder: str,\n",
    "    runtime_files: dict[str, Any],\n",
    "    export_report: dict[str, Any],\n",
    ") -> None:\n",
    "    manifest = {\n",
    "        \"name\": \"Voz_Noslen F5-TTS ONNX/Lite package\",\n",
    "        \"created_at_utc\": datetime.now(timezone.utc).isoformat(),\n",
    "        \"packager_version\": PACKAGER_VERSION,\n",
    "        \"source\": source_label,\n",
    "        \"source_revision\": revision,\n",
    "        \"target_huggingface_folder\": hf_folder,\n",
    "        \"f5_tts_model\": \"F5TTS_v1_Base\",\n",
    "        \"sample_rate\": 24000,\n",
    "        \"vocoder\": \"vocos\",\n",
    "        \"runtime_files\": runtime_files,\n",
    "        \"onnx_files\": sorted(path.relative_to(paths.staging_root).as_posix() for path in paths.onnx_dir.glob(\"*.onnx\")),\n",
    "        \"test_script\": \"scripts/test_package_cpu.py\",\n",
    "        \"lite_contract_status\": \"partial_pipeline\",\n",
    "        \"lite_contract\": {\n",
    "            \"available_onnx\": \"DiT/Transformer core only: inputs x, cond, text, time, mask; output pred.\",\n",
    "            \"full_text_to_audio_onnx\": False,\n",
    "            \"required_runtime\": \"Python preprocessing/sampling/postprocessing from f5-tts plus vocos runtime.\",\n",
    "        },\n",
    "        \"limitations\": [\n",
    "            \"F5-TTS inference includes tokenizer/preprocess, reference-audio conditioning, iterative flow-matching sampling, vocoder and WAV writing.\",\n",
    "            \"The exported ONNX is not a single text-to-waveform graph and cannot satisfy a high-level text/text_ids -> audio backend contract by itself.\",\n",
    "            \"The CPU WAV test uses f5-tts Python + vocos for the full pipeline and onnxruntime only to validate the exported DiT/core graph.\",\n",
    "            \"Reference text is used when present; otherwise F5-TTS preprocessing may invoke automatic transcription for the reference audio.\",\n",
    "        ],\n",
    "        \"onnx_export_summary\": export_report,\n",
    "    }\n",
    "    paths.root_manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n",
    "\n",
    "\n",
    "def list_package_files(paths: PackagePaths) -> list[dict[str, Any]]:\n",
    "    files: list[dict[str, Any]] = []\n",
    "    for path in sorted(item for item in paths.staging_root.rglob(\"*\") if item.is_file()):\n",
    "        files.append(\n",
    "            {\n",
    "                \"path\": path.relative_to(paths.staging_root).as_posix(),\n",
    "                \"size_bytes\": path.stat().st_size,\n",
    "            }\n",
    "        )\n",
    "    return files\n",
    "\n",
    "\n",
    "def run_cpu_package_test(paths: PackagePaths, test_text: str, nfe_step_value: int, speed_value: float) -> dict[str, Any]:\n",
    "    command = [\n",
    "        sys.executable,\n",
    "        str(paths.scripts_dir / \"test_package_cpu.py\"),\n",
    "        \"--package-dir\",\n",
    "        str(paths.staging_root),\n",
    "        \"--text\",\n",
    "        test_text,\n",
    "        \"--output-wav\",\n",
    "        \"test_outputs/voz_noslen_lite_cpu.wav\",\n",
    "        \"--nfe-step\",\n",
    "        str(nfe_step_value),\n",
    "        \"--speed\",\n",
    "        str(speed_value),\n",
    "    ]\n",
    "    LOGGER.info(\"Rodando teste CPU do pacote: %s\", \" \".join(command))\n",
    "    start = datetime.now(timezone.utc)\n",
    "    completed = subprocess.run(command, cwd=str(paths.staging_root), text=True, capture_output=True, check=False)\n",
    "    elapsed = (datetime.now(timezone.utc) - start).total_seconds()\n",
    "    result = {\n",
    "        \"command\": \" \".join(command),\n",
    "        \"elapsed_seconds\": elapsed,\n",
    "        \"returncode\": completed.returncode,\n",
    "        \"stdout_tail\": completed.stdout[-4000:],\n",
    "        \"stderr_tail\": completed.stderr[-4000:],\n",
    "    }\n",
    "    if completed.returncode != 0:\n",
    "        raise RuntimeError(f\"Teste CPU falhou com codigo {completed.returncode}. stderr: {completed.stderr[-2000:]}\")\n",
    "    return result\n",
    "\n",
    "\n",
    "def write_package_metadata(\n",
    "    paths: PackagePaths,\n",
    "    repo_id: str,\n",
    "    revision: str,\n",
    "    hf_folder: str,\n",
    "    checkpoint_path: Path,\n",
    "    vocab_path: Path,\n",
    "    reference_audio_path: Path | None,\n",
    "    manifest_path: Path | None,\n",
    "    manifest: dict[str, Any] | None,\n",
    "    export_report: dict[str, Any],\n",
    ") -> None:\n",
    "    metadata = {\n",
    "        \"created_at_utc\": datetime.now(timezone.utc).isoformat(),\n",
    "        \"source_repo_id\": repo_id,\n",
    "        \"source_revision\": revision,\n",
    "        \"target_huggingface_folder\": hf_folder,\n",
    "        \"policy\": \"Arquivos originais copiados; nenhum arquivo do treinamento remoto e alterado.\",\n",
    "        \"copied_training_dir\": str(paths.copied_training_dir),\n",
    "        \"checkpoint\": str(checkpoint_path.relative_to(paths.copied_training_dir)),\n",
    "        \"vocab\": str(vocab_path.relative_to(paths.copied_training_dir)),\n",
    "        \"reference_audio\": str(reference_audio_path.relative_to(paths.copied_training_dir)) if reference_audio_path else None,\n",
    "        \"manifest\": str(manifest_path.relative_to(paths.copied_training_dir)) if manifest_path else None,\n",
    "        \"manifest_summary\": manifest or {},\n",
    "        \"onnx_export\": export_report,\n",
    "    }\n",
    "    paths.metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n",
    "\n",
    "\n",
    "def upload_package(paths: PackagePaths, repo_id: str, revision: str, hf_folder: str, token: str | None) -> None:\n",
    "    from huggingface_hub import HfApi\n",
    "\n",
    "    if not token:\n",
    "        raise RuntimeError(\"HF_TOKEN ausente. Crie um Kaggle Secret chamado HF_TOKEN para enviar ao Hugging Face.\")\n",
    "    api = HfApi(token=token)\n",
    "    api.create_repo(repo_id=repo_id, repo_type=\"model\", exist_ok=True)\n",
    "    LOGGER.info(\"Enviando pacote para %s/%s\", repo_id, hf_folder)\n",
    "    api.upload_folder(\n",
    "        repo_id=repo_id,\n",
    "        repo_type=\"model\",\n",
    "        revision=revision,\n",
    "        folder_path=str(paths.staging_root),\n",
    "        path_in_repo=hf_folder.strip(\"/\"),\n",
    "        commit_message=f\"Add F5-TTS ONNX package at {hf_folder}\",\n",
    "    )\n",
    "\n",
    "\n",
    "def package_voice(args: argparse.Namespace) -> None:\n",
    "    LOGGER.info(\"Voz_Noslen ONNX packager versao: %s\", PACKAGER_VERSION)\n",
    "    token = get_hf_token()\n",
    "    upload_repo_id = args.upload_repo_id or args.repo_id\n",
    "    revision = args.revision\n",
    "    timestamp = datetime.now(timezone.utc).strftime(\"%Y%m%d_%H%M%S\")\n",
    "    hf_folder = args.hf_folder or f\"onnx_packages/voz_noslen_f5tts_onnx_{timestamp}\"\n",
    "\n",
    "    source, source_label = download_source(args.source, args.repo_id, revision, token, args.voice_dir, args.download_mode)\n",
    "    paths = make_package_paths()\n",
    "    move_tree(source, paths.copied_training_dir)\n",
    "\n",
    "    manifest_path = find_manifest(paths.copied_training_dir)\n",
    "    manifest = load_json_if_exists(manifest_path)\n",
    "    checkpoint_path = find_checkpoint(paths.copied_training_dir, manifest, manifest_path)\n",
    "    vocab_path = find_vocab(paths.copied_training_dir, checkpoint_path)\n",
    "    reference_audio_path = find_reference_audio(paths.copied_training_dir, args.voice_dir)\n",
    "    reference_text = extract_reference_text(manifest, manifest_path, reference_audio_path)\n",
    "\n",
    "    LOGGER.info(\"Checkpoint escolhido: %s\", checkpoint_path)\n",
    "    LOGGER.info(\"Vocab escolhido: %s\", vocab_path)\n",
    "    LOGGER.info(\"Referencia de audio: %s\", reference_audio_path or \"nao encontrada\")\n",
    "    LOGGER.info(\"Texto de referencia: %s\", \"encontrado\" if reference_text else \"nao encontrado\")\n",
    "\n",
    "    if not reference_audio_path:\n",
    "        raise FileNotFoundError(\"Audio de referencia obrigatorio nao encontrado; pacote Lite nao pode ser testado.\")\n",
    "\n",
    "    runtime_files = copy_required_runtime_files(paths, checkpoint_path, vocab_path, reference_audio_path, reference_text)\n",
    "    test_script_path = write_cpu_test_script(paths)\n",
    "\n",
    "    export_report: dict[str, Any]\n",
    "    try:\n",
    "        export_report = export_f5_core_to_onnx(checkpoint_path, vocab_path, paths.onnx_dir, manifest)\n",
    "    except Exception as exc:\n",
    "        export_report = {\n",
    "            \"status\": \"failed\",\n",
    "            \"packager_version\": PACKAGER_VERSION,\n",
    "            \"error_type\": type(exc).__name__,\n",
    "            \"error\": str(exc),\n",
    "            \"traceback\": \"\".join(traceback.format_exception(type(exc), exc, exc.__traceback__)),\n",
    "            \"note\": \"A copia dos arquivos originais foi preservada. Corrija a exportacao antes de usar este pacote como ONNX.\",\n",
    "        }\n",
    "        paths.export_report_path.write_text(json.dumps(export_report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n",
    "        if not args.allow_failed_export:\n",
    "            LOGGER.error(\"Exportacao ONNX falhou. Relatorio: %s\", paths.export_report_path)\n",
    "            raise\n",
    "\n",
    "    export_report[\"pipeline_contract\"] = {\n",
    "        \"full_text_to_audio_onnx_available\": False,\n",
    "        \"reason\": (\n",
    "            \"O F5-TTS completo depende de preprocessamento/tokenizacao, condicionamento por audio de referencia, \"\n",
    "            \"loop iterativo de flow matching/sampling e vocoder. O arquivo ONNX exportado cobre apenas o nucleo DiT.\"\n",
    "        ),\n",
    "        \"backend_lite_compatibility\": \"Nao compativel com um backend que espera um unico ONNX text/text_ids -> waveform.\",\n",
    "        \"documented_pipeline\": [\n",
    "            \"1. preprocess/tokenizer em Python via f5-tts\",\n",
    "            \"2. DiT/Transformer core ONNX para validacao isolada do nucleo\",\n",
    "            \"3. sampling F5-TTS em Python\",\n",
    "            \"4. vocoder vocos em runtime Python\",\n",
    "            \"5. escrita WAV por soundfile\",\n",
    "        ],\n",
    "    }\n",
    "    export_report[\"test_script\"] = test_script_path.relative_to(paths.staging_root).as_posix()\n",
    "    export_report[\"required_runtime_files\"] = runtime_files\n",
    "    paths.export_report_path.write_text(json.dumps(export_report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n",
    "    write_root_manifest(paths, source_label, revision, hf_folder, runtime_files, export_report)\n",
    "\n",
    "    cpu_test_result: dict[str, Any] | None = None\n",
    "    if args.run_cpu_test:\n",
    "        try:\n",
    "            cpu_test_result = run_cpu_package_test(paths, args.test_text, args.test_nfe_step, args.test_speed)\n",
    "            export_report = load_json_if_exists(paths.export_report_path) or export_report\n",
    "            export_report[\"packager_cpu_test\"] = cpu_test_result\n",
    "        except Exception as exc:\n",
    "            export_report[\"packager_cpu_test\"] = {\n",
    "                \"status\": \"failed\",\n",
    "                \"error_type\": type(exc).__name__,\n",
    "                \"error\": str(exc),\n",
    "                \"note\": \"Pacote nao deve ser publicado como validado ate este teste passar em CPU.\",\n",
    "            }\n",
    "            paths.export_report_path.write_text(json.dumps(export_report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n",
    "            if not args.allow_failed_cpu_test:\n",
    "                raise\n",
    "    else:\n",
    "        export_report[\"packager_cpu_test\"] = {\n",
    "            \"status\": \"skipped\",\n",
    "            \"note\": \"Use scripts/test_package_cpu.py para validar onnxruntime CPU e gerar WAV antes de publicar.\",\n",
    "        }\n",
    "\n",
    "    export_report[\"generated_files\"] = list_package_files(paths)\n",
    "    paths.export_report_path.write_text(json.dumps(export_report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n",
    "    write_root_manifest(paths, source_label, revision, hf_folder, runtime_files, export_report)\n",
    "    write_package_metadata(\n",
    "        paths,\n",
    "        source_label,\n",
    "        revision,\n",
    "        hf_folder,\n",
    "        checkpoint_path,\n",
    "        vocab_path,\n",
    "        reference_audio_path,\n",
    "        manifest_path,\n",
    "        manifest,\n",
    "        export_report,\n",
    "    )\n",
    "\n",
    "    if args.upload:\n",
    "        if not upload_repo_id:\n",
    "            raise RuntimeError(\n",
    "                \"Para fazer upload, informe --upload-repo-id com o Model Repo de destino. \"\n",
    "                \"Buckets nao aceitam upload via HfApi.upload_folder.\"\n",
    "            )\n",
    "        upload_package(paths, upload_repo_id, revision, hf_folder, token)\n",
    "    else:\n",
    "        LOGGER.info(\"Upload desativado. Pacote local pronto em %s\", paths.staging_root)\n",
    "\n",
    "    LOGGER.info(\"Pacote final: %s\", paths.staging_root)\n",
    "    LOGGER.info(\"Pasta alvo no Hugging Face: %s\", hf_folder)\n",
    "\n",
    "\n",
    "def parse_args(argv: list[str]) -> argparse.Namespace:\n",
    "    parser = argparse.ArgumentParser(description=\"Empacota uma voz F5-TTS em um pacote ONNX no Kaggle.\")\n",
    "    parser.add_argument(\"--source\", default=os.environ.get(\"HF_SOURCE_URL\", DEFAULT_SOURCE_URL), help=\"URL ou repo_id do Hugging Face.\")\n",
    "    parser.add_argument(\"--repo-id\", default=os.environ.get(\"HF_SOURCE_REPO_ID\"), help=\"Repo ID explicito, ex: warllem/Voz_Noslen.\")\n",
    "    parser.add_argument(\"--upload-repo-id\", default=os.environ.get(\"HF_UPLOAD_REPO_ID\"), help=\"Model Repo onde a pasta nova sera enviada.\")\n",
    "    parser.add_argument(\"--voice-dir\", default=os.environ.get(\"HF_VOICE_DIR\", DEFAULT_VOICE_DIR), help=\"Pasta da voz dentro do bucket/repo.\")\n",
    "    parser.add_argument(\n",
    "        \"--download-mode\",\n",
    "        choices=(\"essential\", \"all\"),\n",
    "        default=os.environ.get(\"HF_DOWNLOAD_MODE\", \"essential\"),\n",
    "        help=\"essential baixa apenas a voz escolhida e evita checkpoints duplicados; all tenta baixar tudo.\",\n",
    "    )\n",
    "    parser.add_argument(\"--revision\", default=os.environ.get(\"HF_REVISION\", DEFAULT_REVISION))\n",
    "    parser.add_argument(\"--hf-folder\", default=os.environ.get(\"HF_TARGET_FOLDER\"), help=\"Nova pasta dentro do repo Hugging Face.\")\n",
    "    parser.add_argument(\"--upload\", action=\"store_true\", default=os.environ.get(\"HF_UPLOAD\", \"1\") == \"1\")\n",
    "    parser.add_argument(\"--no-upload\", action=\"store_false\", dest=\"upload\")\n",
    "    parser.add_argument(\n",
    "        \"--allow-failed-export\",\n",
    "        action=\"store_true\",\n",
    "        help=\"Ainda cria e envia o pacote copiado se a exportacao ONNX falhar. Nao recomendado para pacote final.\",\n",
    "    )\n",
    "    parser.add_argument(\"--test-text\", default=os.environ.get(\"F5_ONNX_TEST_TEXT\", DEFAULT_TEST_TEXT))\n",
    "    parser.add_argument(\"--test-nfe-step\", type=int, default=int(os.environ.get(\"F5_ONNX_TEST_NFE_STEP\", \"4\")))\n",
    "    parser.add_argument(\"--test-speed\", type=float, default=float(os.environ.get(\"F5_ONNX_TEST_SPEED\", \"1.0\")))\n",
    "    parser.add_argument(\"--run-cpu-test\", action=\"store_true\", default=os.environ.get(\"F5_ONNX_RUN_CPU_TEST\", \"1\") == \"1\")\n",
    "    parser.add_argument(\"--skip-cpu-test\", action=\"store_false\", dest=\"run_cpu_test\")\n",
    "    parser.add_argument(\n",
    "        \"--allow-failed-cpu-test\",\n",
    "        action=\"store_true\",\n",
    "        help=\"Permite criar o pacote mesmo se o teste CPU falhar. Nao use para publicacao final validada.\",\n",
    "    )\n",
    "    return parser.parse_args(argv)\n",
    "\n",
    "\n",
    "def main(argv: list[str] | None = None) -> None:\n",
    "    args = parse_args(argv or sys.argv[1:])\n",
    "    package_voice(args)\n",
    "\n",
    "\n",
    "if __name__ == \"__main__\":\n",
    "    main()\n"
]

output_path = Path("/kaggle/working/f5_tts_onnx_packager_kaggle.py")
output_path.write_text("".join(script_data), encoding="utf-8")
print(f"Script criado com sucesso em: {output_path} (Versão: 2026.06.16.4)")

In [ ]:
# 3) Configurações de Origem e Destino
import os
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except: 
    print("AVISO: HF_TOKEN não encontrado no Kaggle Secrets. O upload falhará.")

os.environ["HF_SOURCE_URL"] = "https://huggingface.co/buckets/warllem/Voz_Noslen"
os.environ["HF_VOICE_DIR"] = "voices/v_minha_voz_f5_tts_ptbr"
os.environ["HF_UPLOAD_REPO_ID"] = "warllem/Voz_Noslen_ONNX"
os.environ["HF_TARGET_FOLDER"] = "onnx_packages/turbo_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
print("Configurações carregadas.")

In [ ]:
# 4) Execução do Empacotador Turbo
import subprocess, sys
print("Iniciando exportação End-to-End + Quantização INT8...")
process = subprocess.Popen([sys.executable, "f5_tts_onnx_packager_kaggle.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout: 
    print(line, end="")